# Section 2 -- Inspecting an A2A agent

**Before you run anything:** the A2A server must already be up.

```bash
./scripts/run_a2a_server.sh
```

Then run the cells below top to bottom.

### The idea in one line

MCP connected an agent to *tools*. A2A connects an agent to *another agent*.

That difference is not cosmetic:

| | A tool | A peer agent |
|---|---|---|
| What it is | a function | its own model, instructions and tools |
| What you do | **call** it | **delegate** to it |
| You get back | a return value | a task it worked on |

### What we will do

| Step | We do | Question it answers |
|---|---|---|
| 1 | GET the agent card | who are you? |
| 2 | read three fields | what can you do, and where do I send work? |
| 3 | POST `message/send` | here, handle this |
| 4 | watch the dev UI | what does this look like when a model drives it? |

Every section ends with a **What just happened** note.


In [1]:
import json
import httpx

BASE = "http://localhost:8001/a2a/billing_agent"

# a2a-sdk 0.3.x served the card at /.well-known/agent.json; 1.x serves it at
# /.well-known/agent-card.json. In real code use the ADK constant
# AGENT_CARD_WELL_KNOWN_PATH rather than hardcoding either.
CARD_URL = f"{BASE}/.well-known/agent-card.json"
print(CARD_URL)

http://localhost:8001/a2a/billing_agent/.well-known/agent-card.json


## 1. The agent card

This is all of A2A discovery: **a JSON file at a well-known URL.**

No registry, no service mesh, no broker. If you can GET this file, you can work
with the agent.

The next cell just downloads it and prints it. Skim it — we pull out the fields
that matter in step 2.


In [2]:
card = httpx.get(CARD_URL).json()
print(json.dumps(card, indent=2))

{
  "name": "billing_agent",
  "description": "Billing specialist for the IT support desk. Handles invoice questions, duplicate or unexpected charges, refunds, and plan changes.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8001/a2a/billing_agent",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "0.3.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "billing_enquiry",
      "name": "Billing Enquiry",
      "description": "Investigate invoice questions, duplicate charges, refunds and credits, and explain the resolution path.",
      "tags": [
        "billing",
        "invoice",
        "refund",
        "charge",
        "finance"
      ]
    }
  ],
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "url": "http://localhost:8001/a2a/billing_agent"

> **What just happened**
>
> A plain `GET` returned everything needed to work with an agent we have never
> talked to before. That file is served straight from
> `src/helpdesk/a2a/remote/billing_agent/agent.json`.
>
> **Why it matters:** discovery is a static file. No client library was needed
> to read it, and any language that can do HTTP can participate.
>
> **The failure you will actually hit:** no `agent.json`, or it is misnamed. The
> A2A server then publishes nothing and fails silently — the agent simply is
> not there.


## 2. Reading the card the way a caller does

Three fields carry the weight:

- **`skills`** -- what it can do, in natural language. This is what a calling
  agent's model reads to decide whether to delegate here. Same lesson as MCP
  docstrings: vague description, no delegation.
- **`capabilities`** -- protocol-level features, e.g. streaming.
- **`url`** -- where to actually send work.

In [3]:
# An A2A agent card is served in TWO shapes at once, merged into one document.
#
#   legacy (protocol 0.3):  a top-level "url" plus "preferredTransport"
#   current (protocol 1.0): a "supportedInterfaces" list, each entry carrying
#                           its own url + protocolBinding + protocolVersion
#
# The server emits both so that old and new clients can each find what they
# expect. Read the modern field first and fall back -- that is what a real
# client library does for you.

def resolve_rpc_url(card: dict) -> str:
    interfaces = card.get("supportedInterfaces") or []
    if interfaces and interfaces[0].get("url"):
        return interfaces[0]["url"]
    return card["url"]  # legacy 0.3 placement


RPC_URL = resolve_rpc_url(card)

print("name:  ", card["name"])
print("rpc:   ", RPC_URL)
print("caps:  ", card.get("capabilities"))
print()
for skill in card.get("skills", []):
    print(f"- {skill['id']}: {skill['description']}")
    print(f"  tags: {', '.join(skill.get('tags', []))}")


name:   billing_agent
rpc:    http://localhost:8001/a2a/billing_agent
caps:   {'streaming': False, 'pushNotifications': False}

- billing_enquiry: Investigate invoice questions, duplicate charges, refunds and credits, and explain the resolution path.
  tags: billing, invoice, refund, charge, finance


> **What just happened**
>
> We pulled out the three fields a caller actually needs, and resolved the RPC
> URL from the modern field with a fallback to the legacy one.
>
> **Why it matters:** `skills` is the A2A equivalent of an MCP docstring. It is
> what a calling agent's model reads to decide whether to delegate here. Same
> lesson as Section 1 — vague description, no delegation.
>
> **Watch the `url` field.** It is absolute. That is why this workshop keeps all
> traffic inside the Codespace: route it through a public forwarded URL and the
> card still advertises `localhost`, so discovery succeeds and every call fails.
> A genuine deployment gotcha, not a workshop limitation.


## 3. Sending work over the wire

The card told us the agent exists and where it lives. Now we send it work.

**Like an MCP tool call, an A2A request is two things:**

1. a **method** — always `message/send`, no matter what the agent does
2. a **message** — a role, an id, and your text

The next cell builds that JSON-RPC request by hand and POSTs it to the RPC URL
from the card. No SDK, no client library.

Notice what is *not* in the request: there is no `billing` method. Every A2A
agent answers the same call.


In [5]:
import uuid

QUESTION = "What is the refund policy for annual plans?"

payload = {
    "jsonrpc": "2.0",
    "id": str(uuid.uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "messageId": str(uuid.uuid4()),
            "parts": [{"kind": "text", "text": QUESTION}],
        }
    },
}

response = httpx.post(RPC_URL, json=payload, timeout=120.0)
print("status:", response.status_code)


def text_parts(node, seen=None):
    """Collect the agent's text, wherever it sits in the reply."""
    seen = {} if seen is None else seen
    if isinstance(node, dict):
        if node.get("role") == "user":
            return  # the task history echoes our own question back; skip it
        text = node.get("text")
        if node.get("kind") == "text" and text and text not in seen:
            seen[text] = True
            yield text
        for value in node.values():
            yield from text_parts(value, seen)
    elif isinstance(node, list):
        for value in node:
            yield from text_parts(value, seen)


body = response.json()

print("\nasked:", QUESTION)
print("\nthe agent replied:\n")
# The answer arrives in "artifacts"; "history" repeats it, hence the dedupe.
for part in text_parts(body.get("result", {})):
    print(part)

print("\n--- raw envelope (truncated) ---")
print(json.dumps(body, indent=2)[:1200])


status: 200

asked: What is the refund policy for annual plans?

the agent replied:

The knowledge base does not specify a general refund policy for annual plans. It only confirms that duplicate or mid-cycle plan-change charges may receive a prorated credit; approved refunds to the original payment method take **5–7 days**.

**Next step:** Escalate the annual-plan refund request to Finance for eligibility review. Expect the payment refund within **5–7 days after approval**.

--- raw envelope (truncated) ---
{
  "id": "e232a4b3-c58d-4a2b-aaf6-7cb5a67875f9",
  "jsonrpc": "2.0",
  "result": {
    "artifacts": [
      {
        "artifactId": "5b210ea0-ef26-46b8-8dbf-db4c7458a7a9",
        "parts": [
          {
            "kind": "text",
            "text": "The knowledge base does not specify a general refund policy for annual plans. It only confirms that duplicate or mid-cycle plan-change charges may receive a prorated credit; approved refunds to the original payment method take **5\u20

> **What just happened**
>
> We sent one sentence over HTTP and a *different process* — with its own model
> and its own instructions — thought about it and answered. This is byte-for-byte
> what `triage_agent` sends when it hands a billing question to `billing_agent`.
>
> **Why it matters:** the specialty lives in the **card**, not in the method
> names. Every agent exposes the identical `message/send` verb, so adding a
> tenth agent to your system requires no new client code. That uniformity is
> what makes agents composable.
>
> **Note the reply shape.** It came back as a *task*, not just a string —
> because delegation can be long-running. Compare that to MCP, where
> `call_tool()` returns a value and is done.


## 4. Now watch an agent do it

So far *we* have been the caller. Now let a model do it.

Start the dev UI in a second terminal:

```bash
./scripts/run_web.sh
```

Open http://127.0.0.1:8002, pick **triage_agent**, and ask:

> *Why was I charged twice this month?*

Watch the trace. Triage decides this is billing and hands off to
`billing_agent` — a different process, reached over HTTP, that triage knows
about only through the card you read above.

Then open `src/helpdesk/a2a/local/triage_agent/agent.py`. The remote agent is
declared in three lines and dropped into `sub_agents=[...]` beside local ones.

> **What just happened**
>
> From the model's point of view there was no difference between a local
> sub-agent and one running in another process on another machine.
>
> **Why it matters:** that is the whole abstraction A2A buys you. Teams can ship
> agents independently, in different languages, on different infrastructure, and
> still compose them.

---

## Recap

| We did | The lesson |
|---|---|
| GET the agent card | discovery is a static JSON file at a known URL |
| Read `skills` | it is the prompt another model reads — vague means ignored |
| Read `url` | it is absolute, which breaks under URL forwarding |
| POST `message/send` | one verb for every agent; specialty lives in the card |
| Watched the UI | remote agents look local to the model |

Next: `docs/03-lab-interop.md`.
